# Đồ án phân loại ảnh động vật với CNN — TensorFlow/Keras

**Mục tiêu:** phân loại 10 loài trong [Animals-10](https://www.kaggle.com/datasets/alessiocorrado99/animals10), xây dựng CNN cơ sở và so sánh với CNN ResNet50 học chuyển giao. Toàn bộ mô hình, tensor, pipeline dữ liệu và huấn luyện dùng TensorFlow/Keras.

Chạy lần lượt từ trên xuống: khảo sát dữ liệu → chia train/validation/test → tiền xử lý → xây dựng, huấn luyện → đánh giá → phân tích lỗi → Grad-CAM → demo. Chọn kernel `.venv` sau khi cài `requirements.txt`, rồi **Restart Kernel**. Lần đầu tải dataset và trọng số ImageNet cần Internet; huấn luyện đầy đủ trên CPU có thể rất lâu.

Windows chạy trực tiếp dùng CPU. Để dùng GPU NVIDIA, mở dự án bằng VS Code trong WSL2 và cài TensorFlow với extra `and-cuda` theo README. Ô cấu hình in thiết bị thực tế.

Đây là mã và khung phân tích, không phải kết quả đã đo. Không điền accuracy giả định vào báo cáo. Kết quả mới lưu riêng trong `runs/tensorflow/`; trọng số `.pt` cũ không dùng được, cần huấn luyện lại.

## 1. Thư viện và cấu hình

TensorFlow xử lý tensor và pipeline `tf.data`; Keras cung cấp các lớp CNN, ResNet50, optimizer và `model.fit()`. scikit-learn chia dữ liệu và tính chỉ số. `DATA_DIR` phải trỏ trực tiếp tới `raw-img`; bật `DOWNLOAD_DATASET` để tải qua kagglehub. Không đặt dữ liệu hay trọng số trong Git.

Chọn các tham số trước khi bắt đầu. Random seed giúp lặp lại phép chia và khởi tạo; phần cứng và phiên bản thư viện vẫn có thể tạo sai khác nhỏ.

In [ ]:
import csv, json
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps, UnidentifiedImageError
from IPython.display import display, Image as DisplayImage
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

LABELS = {"cane": "Chó", "cavallo": "Ngựa", "elefante": "Voi", "farfalla": "Bướm", "gallina": "Gà", "gatto": "Mèo", "mucca": "Bò", "pecora": "Cừu", "ragno": "Nhện", "scoiattolo": "Sóc"}
CLASSES = sorted(LABELS)
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMAGE_SIZE = (224, 224)

DOWNLOAD_DATASET = True  # Đổi thành False nếu đã tải dataset về.
DATA_DIR = Path("data/raw-img")
RUNS = Path("runs/tensorflow")
SEED = 42
BATCH_SIZE = 8  # Giảm xuống 4 nếu thiếu bộ nhớ.
CUSTOM_EPOCHS = 15
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 10
USE_CLASS_WEIGHTS = False

GPUS = tf.config.list_physical_devices("GPU")
for gpu in GPUS:
    tf.config.experimental.set_memory_growth(gpu, True)
print("TensorFlow:", tf.__version__)
print("Thiết bị:", GPUS if GPUS else "CPU (GPU NVIDIA trên Windows cần WSL2)")

In [ ]:
def seed_everything(seed):
    keras.utils.set_random_seed(seed)

In [ ]:
seed_everything(SEED)
if DOWNLOAD_DATASET:
    import kagglehub
    downloaded = Path(kagglehub.dataset_download("alessiocorrado99/animals10"))
    candidates = [p for p in downloaded.rglob("raw-img") if p.is_dir()]
    if downloaded.name == "raw-img":
        candidates.insert(0, downloaded)
    if len(candidates) != 1:
        raise ValueError(f"Cần đúng một thư mục raw-img, tìm thấy: {candidates}. Hãy đặt DATA_DIR thủ công.")
    DATA_DIR = candidates[0]
DATA_DIR = DATA_DIR.expanduser().resolve()
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy {DATA_DIR}. Sửa DATA_DIR hoặc bật DOWNLOAD_DATASET.")
print("Dataset:", DATA_DIR)

## 2. Khảo sát dữ liệu và chia tập

Tên thư mục gốc là tiếng Ý; bảng `LABELS` ánh xạ sang tiếng Việt. Ta đọc từng ảnh để loại ảnh lỗi, ghi lại kích thước gốc và đếm số lượng theo lớp. Biểu đồ cột giúp phát hiện mất cân bằng; biểu đồ độ phân giải và tỷ lệ chiều rộng/chiều cao cho thấy sự đa dạng ảnh. Histogram giới hạn trục tỷ lệ trong [0, 4]; ảnh ngoài khoảng này không được vẽ trong histogram.

Chia **70% train, 15% validation, 15% test** bằng phân tầng theo nhãn và cùng seed. Train dùng để cập nhật trọng số; validation chọn checkpoint; test chỉ dùng đánh giá sau khi chọn cấu hình. Manifest lưu đường dẫn tương đối để cả hai mô hình dùng cùng ảnh. Chia theo ảnh chưa ngăn các ảnh trùng hoặc gần trùng lọt sang các tập khác nhau; cần rà soát trùng lặp trước khi kết luận về khả năng tổng quát hóa.

In [ ]:
def prepare(args):
    root, out = Path(args.data).resolve(), Path(args.output)
    out.mkdir(parents=True, exist_ok=True)
    missing = [c for c in CLASSES if not (root / c).is_dir()]
    if missing:
        raise ValueError(f"Không tìm thấy thư mục lớp: {missing}. DATA_DIR phải trỏ tới raw-img")
    rows, bad = [], []
    for label in CLASSES:
        for path in sorted((root / label).rglob("*")):
            if path.suffix.lower() not in EXTENSIONS or not path.is_file():
                continue
            try:
                with Image.open(path) as im:
                    im.load()
                    w, h = ImageOps.exif_transpose(im).size
                rows.append({"path": path.relative_to(root).as_posix(), "label": label, "width": w, "height": h})
            except (OSError, ValueError, UnidentifiedImageError) as exc:
                bad.append(f"{path.relative_to(root)}: {exc}")
    counts = Counter(r["label"] for r in rows)
    if any(counts[c] < 7 for c in CLASSES):
        raise ValueError(f"Mỗi lớp cần ít nhất 7 ảnh hợp lệ để chia 70/15/15: {counts}")
    train, rest = train_test_split(rows, test_size=0.3, random_state=args.seed, stratify=[r["label"] for r in rows])
    val, test = train_test_split(rest, test_size=0.5, random_state=args.seed, stratify=[r["label"] for r in rest])
    with (out / "splits.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["path", "label", "width", "height", "split"])
        writer.writeheader()
        for name, group in [("train", train), ("val", val), ("test", test)]:
            writer.writerows([{**row, "split": name} for row in group])
    (out / "invalid_images.txt").write_text("\n".join(bad), encoding="utf-8")
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    axes[0].bar(CLASSES, [counts[c] for c in CLASSES]); axes[0].tick_params(axis="x", rotation=70); axes[0].set_title("Images per class")
    axes[1].scatter([r["width"] for r in rows], [r["height"] for r in rows], s=2, alpha=.2); axes[1].set(xlabel="Width", ylabel="Height", title="Image resolution")
    axes[2].hist([r["width"] / r["height"] for r in rows], bins=50, range=(0, 4)); axes[2].set(xlabel="Width / height (0–4)", title="Aspect ratio")
    fig.tight_layout(); fig.savefig(out / "eda.png", dpi=160); plt.close(fig)
    print(f"Ảnh hợp lệ: {len(rows)}, lỗi: {len(bad)}; train/val/test: {len(train)}/{len(val)}/{len(test)}. Kết quả: {out}")

In [ ]:
prepare(SimpleNamespace(data=DATA_DIR, output=RUNS / "data", seed=SEED))
SPLITS = RUNS / "data" / "splits.csv"
display(DisplayImage(filename=str(RUNS / "data" / "eda.png")))

**Cách phân tích:** ghi lại lớp nhiều/ít ảnh nhất, mức chênh lệch và những kích thước thường gặp từ biểu đồ thực tế. Nếu lớp ít ảnh có recall thấp, thử `USE_CLASS_WEIGHTS = True` trong một lần chạy riêng rồi so sánh macro F1 trên validation. Trọng số lớp chỉ là một lựa chọn, không bảo đảm cải thiện mọi chỉ số.

## 3. Tiền xử lý và pipeline tf.data

Pillow đọc RGB, sửa hướng EXIF, resize 224×224 (bilinear), hỗ trợ cả WebP. `tf.numpy_function` đưa bước đọc này vào `tf.data`; dữ liệu dùng định dạng NHWC `(batch, height, width, channels)`. Resize trực tiếp có thể làm méo ảnh.

Chỉ train có augmentation Keras: lật ngang, xoay tối đa 15°, thay đổi độ sáng và tương phản nhẹ. CNN cơ sở chia pixel cho 255; ResNet50 dùng đúng `keras.applications.resnet50.preprocess_input` (RGB → BGR, trừ mean ImageNet, không chia 255). Validation/test/demo/Grad-CAM dùng cùng hàm tiền xử lý và không augmentation.

In [ ]:
def image_array(image):
    image = ImageOps.exif_transpose(image).convert("RGB")
    return np.asarray(image.resize(IMAGE_SIZE, Image.Resampling.BILINEAR), dtype=np.float32)


def read_image(path):
    with Image.open(path.decode("utf-8")) as image:
        return image_array(image)


def preprocess_images(images, architecture):
    images = tf.cast(images, tf.float32)
    if architecture == "custom":
        return images / 255.0
    if architecture == "resnet50":
        return keras.applications.resnet50.preprocess_input(images)
    raise ValueError(f"Kiến trúc không hỗ trợ: {architecture}")


def read_splits(path):
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not rows or any(r["label"] not in CLASSES or r["split"] not in {"train", "val", "test"} for r in rows):
        raise ValueError("Manifest splits.csv không hợp lệ")
    return rows


def loader(root, rows, split, batch_size, architecture, class_weights=None, seed=SEED):
    selected = [r for r in rows if r["split"] == split]
    if not selected:
        raise ValueError(f"Tập {split} rỗng")
    paths = [str(Path(root) / r["path"]) for r in selected]
    labels = [CLASSES.index(r["label"]) for r in selected]
    data = tf.data.Dataset.from_tensor_slices((paths, labels))
    if split == "train":
        data = data.shuffle(len(selected), seed=seed, reshuffle_each_iteration=True)

    def decode(path, label):
        image = tf.numpy_function(read_image, [path], tf.float32)
        image.set_shape((*IMAGE_SIZE, 3))
        return image, label

    data = data.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size)
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal", seed=seed),
        layers.RandomRotation(15 / 360, fill_mode="reflect", seed=seed + 1),
        layers.RandomBrightness(.15, value_range=(0, 255), seed=seed + 2),
        layers.RandomContrast(.15, seed=seed + 3),
    ]) if split == "train" else None

    def transform(images, labels):
        if augmentation is not None:
            images = tf.clip_by_value(augmentation(images, training=True), 0, 255)
        images = preprocess_images(images, architecture)
        if class_weights is not None:
            return images, labels, tf.gather(class_weights, labels)
        return images, labels

    return data.map(transform, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [ ]:
rows = read_splits(SPLITS)
for split in ("train", "val", "test"):
    counts = Counter(r["label"] for r in rows if r["split"] == split)
    print(split, sum(counts.values()), dict(sorted(counts.items())))
train_data = loader(DATA_DIR, rows, "train", BATCH_SIZE, "custom")
images, labels = next(iter(train_data))
print("Kích thước batch NHWC:", images.shape, "— nhãn:", labels.shape)

## 4. CNN cơ sở tự xây dựng

Mỗi block gồm `Conv2D(3×3, padding="same")` → BatchNormalization → ReLU → MaxPooling2D(2×2). Số kênh tăng 3 → 32 → 64 → 128 → 256, kích thước giảm 224 → 112 → 56 → 28 → 14. GlobalAveragePooling2D, Dropout(0,5) và Dense tạo 10 logits. Loss là SparseCategoricalCrossentropy(from_logits=True); chỉ dùng softmax khi dự đoán.

CNN cơ sở giúp đánh giá giá trị của transfer learning. Accuracy phải dựa trên thực nghiệm.

In [ ]:
def CustomCNN():
    inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
    x = inputs
    for block, channels in enumerate((32, 64, 128, 256), start=1):
        x = layers.Conv2D(channels, 3, padding="same", name=f"conv_{block}")(x)
        x = layers.BatchNormalization(name=f"bn_{block}")(x)
        x = layers.ReLU(name=f"features_{block}")(x)
        x = layers.MaxPooling2D(2, name=f"pool_{block}")(x)
    x = layers.GlobalAveragePooling2D(name="global_pool")(x)
    x = layers.Dropout(.5)(x)
    outputs = layers.Dense(len(CLASSES), name="classifier")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")


def make_model(name, pretrained=False):
    if name == "custom":
        return CustomCNN()
    if name != "resnet50":
        raise ValueError(f"Kiến trúc không hỗ trợ: {name}")
    backbone = keras.applications.ResNet50(
        include_top=False, weights="imagenet" if pretrained else None,
        input_shape=(*IMAGE_SIZE, 3),
    )
    x = layers.GlobalAveragePooling2D(name="global_pool")(backbone.output)
    outputs = layers.Dense(len(CLASSES), name="classifier")(x)
    return keras.Model(backbone.input, outputs, name="animals_resnet50")


def configure_resnet(model, stage):
    # BatchNormalization đóng băng luôn dùng thống kê inference của ImageNet.
    for layer in model.layers:
        layer.trainable = layer.name == "classifier" or (
            stage == "finetune" and layer.name.startswith("conv5_")
            and not isinstance(layer, layers.BatchNormalization)
        )

In [ ]:
baseline = CustomCNN()
baseline.summary()
print("Số tham số:", baseline.count_params())
print("Đầu ra batch giả lập:", baseline(tf.zeros((1, *IMAGE_SIZE, 3)), training=False).shape)

## 5. Huấn luyện và chọn checkpoint Keras

`model.compile()` cấu hình AdamW, sparse cross-entropy và accuracy; `model.fit()` huấn luyện và đánh giá validation mỗi epoch. `ModelCheckpoint` giữ `best.keras` có **validation loss thấp nhất** xuyên suốt các giai đoạn. `metadata.json` lưu kiến trúc, thứ tự lớp và kích thước ảnh để kiểm tra khi nạp.

Nếu bật trọng số lớp, mỗi mẫu train và validation nhận trọng số nghịch với số lượng ảnh lớp đó trong train; loss là trung bình loss có trọng số trên số mẫu, accuracy vẫn không trọng số. Test luôn đánh giá không trọng số. So sánh mô hình với cùng manifest và cấu hình loss.

Train loss giảm nhưng validation loss tăng có thể là overfitting. Augmentation có thể khiến train accuracy thấp hơn validation accuracy.

In [ ]:
def train(args):
    seed_everything(args.seed)
    stages = [("scratch", args.epochs, args.lr)] if args.model == "custom" else [
        ("head", args.head_epochs, args.lr), ("finetune", args.finetune_epochs, args.finetune_lr),
    ]
    if any(epochs < 0 for _, epochs, _ in stages) or not any(epochs > 0 for _, epochs, _ in stages):
        raise ValueError("Cần ít nhất một epoch, số epoch không được âm")
    rows = read_splits(args.splits)
    counts = Counter(r["label"] for r in rows if r["split"] == "train")
    if any(not counts[c] for c in CLASSES):
        raise ValueError("Tập train phải chứa đủ 10 lớp")
    weights = tf.constant([sum(counts.values()) / (len(CLASSES) * counts[c]) for c in CLASSES], tf.float32) if args.class_weights else None
    train_data = loader(args.data, rows, "train", args.batch_size, args.model, weights, seed=args.seed)
    val_data = loader(args.data, rows, "val", args.batch_size, args.model, weights, seed=args.seed)
    model = make_model(args.model, pretrained=args.model == "resnet50")
    out = Path(args.output)
    out.mkdir(parents=True, exist_ok=True)
    checkpoint = keras.callbacks.ModelCheckpoint(
        str(out / "best.keras"), monitor="val_loss", mode="min", save_best_only=True,
    )
    history = []
    for stage, epochs, lr in stages:
        if not epochs:
            continue
        if args.model == "resnet50":
            if stage == "finetune" and history:
                best_model = keras.models.load_model(out / "best.keras", compile=False)
                model.set_weights(best_model.get_weights())
                del best_model
            configure_resnet(model, stage)
        # Recompile sau khi thay đổi trainable và tạo optimizer cho giai đoạn mới.
        model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=.01),
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
        )
        print(f"Giai đoạn {stage}: {epochs} epoch, learning rate={lr}")
        result = model.fit(train_data, validation_data=val_data, epochs=epochs,
                           callbacks=[checkpoint], shuffle=False, verbose=2)
        for i in range(len(result.history["loss"])):
            history.append({"stage": stage, "epoch": len(history) + 1,
                            "train_loss": float(result.history["loss"][i]),
                            "train_acc": float(result.history["accuracy"][i]),
                            "val_loss": float(result.history["val_loss"][i]),
                            "val_acc": float(result.history["val_accuracy"][i])})
    if not np.isfinite(checkpoint.best):
        raise RuntimeError("Validation loss không hữu hạn; kiểm tra dữ liệu và learning rate")
    metadata = {"architecture": args.model, "classes": CLASSES, "image_size": list(IMAGE_SIZE),
                "tensorflow_version": tf.__version__}
    (out / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    (out / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["loss", "acc"]):
        for split in ["train", "val"]:
            ax.plot([r["epoch"] for r in history], [r[f"{split}_{metric}"] for r in history], label=split)
        ax.set(xlabel="Epoch", ylabel=metric)
        ax.legend()
    fig.tight_layout()
    fig.savefig(out / "learning_curves.png", dpi=160)
    plt.close(fig)

In [ ]:
common = dict(data=DATA_DIR, splits=SPLITS, batch_size=BATCH_SIZE,
              seed=SEED, class_weights=USE_CLASS_WEIGHTS, lr=1e-3, finetune_lr=1e-4,
              epochs=CUSTOM_EPOCHS, head_epochs=HEAD_EPOCHS, finetune_epochs=FINETUNE_EPOCHS)
train(SimpleNamespace(**common, model="custom", output=RUNS / "custom"))
display(DisplayImage(filename=str(RUNS / "custom" / "learning_curves.png")))

## 6. Transfer learning với CNN ResNet50

ResNet50 là mạng CNN với kết nối residual, khởi tạo bằng trọng số ImageNet. Giai đoạn 1 đóng băng backbone, chỉ huấn luyện `classifier` với learning rate 0,001. Giai đoạn 2 nạp trọng số tốt nhất của head, mở các lớp tích chập nhóm `conv5_*`, giảm learning rate xuống 0,0001 và compile lại. Các lớp BatchNormalization luôn đóng băng để giữ thống kê ImageNet khi fine-tune với batch nhỏ, theo [hướng dẫn Keras](https://keras.io/guides/transfer_learning/).

Một callback checkpoint dùng xuyên suốt hai giai đoạn, nên mô hình cuối là mô hình có validation loss thấp nhất của cả hai. Lần đầu cần Internet tải trọng số. Kiến thức học từ ImageNet là lợi thế dữ liệu bên ngoài so với CNN học từ đầu.

In [ ]:
train(SimpleNamespace(**common, model="resnet50", output=RUNS / "resnet50"))
display(DisplayImage(filename=str(RUNS / "resnet50" / "learning_curves.png")))

## 7. Đánh giá trên tập test

Accuracy là tỷ lệ dự đoán đúng; precision cho biết trong các dự đoán một lớp có bao nhiêu dự đoán đúng; recall cho biết mô hình tìm được bao nhiêu ảnh thực sự thuộc lớp đó. F1 là trung bình điều hòa của precision và recall. Macro F1 coi các lớp quan trọng ngang nhau; weighted F1 tính theo số mẫu, do đó lớp lớn ảnh hưởng nhiều hơn.

Ma trận nhầm lẫn có **hàng là nhãn thật, cột là nhãn dự đoán**. Các ô ngoài đường chéo cho thấy cặp lớp hay bị nhầm. Chỉ chạy ô test sau khi đã chọn cấu hình dựa trên validation; không tối ưu lại theo test. So sánh hai kiến trúc là một báo cáo cuối cùng, không dùng tập test để tiếp tục chọn siêu tham số.

In [ ]:
def load_checkpoint(path):
    path = Path(path)
    if path.suffix != ".keras":
        raise ValueError("Cần checkpoint .keras được huấn luyện bằng notebook TensorFlow")
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {path}. Hãy chạy ô huấn luyện trước.")
    metadata = json.loads(path.with_name("metadata.json").read_text(encoding="utf-8"))
    if metadata["classes"] != CLASSES or metadata["image_size"] != list(IMAGE_SIZE):
        raise ValueError("Thứ tự lớp hoặc kích thước ảnh trong checkpoint không khớp")
    architecture = metadata["architecture"]
    if architecture not in {"custom", "resnet50"}:
        raise ValueError(f"Kiến trúc không hỗ trợ: {architecture}")
    model = keras.models.load_model(path, compile=False)
    if model.output_shape[-1] != len(CLASSES):
        raise ValueError("Số đầu ra của checkpoint không khớp số lớp")
    return model, architecture


def predict_probabilities(model, architecture, image):
    batch = preprocess_images(image_array(image)[None, ...], architecture)
    return tf.nn.softmax(model(batch, training=False), axis=-1)[0].numpy()


def gradcam(model, architecture, image):
    target_layer = "conv5_block3_out" if architecture == "resnet50" else "features_4"
    gradient_model = keras.Model(model.inputs, [model.get_layer(target_layer).output, model.output])
    batch = preprocess_images(image_array(image)[None, ...], architecture)
    with tf.GradientTape() as tape:
        # Theo dõi input cả khi backbone đã đóng băng hoàn toàn.
        tape.watch(batch)
        features, logits = gradient_model(batch, training=False)
        score = logits[0, tf.argmax(logits[0])]
    gradients = tape.gradient(score, features)
    if gradients is None:
        raise RuntimeError("Không tính được gradient cho Grad-CAM")
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heat = tf.nn.relu(tf.reduce_sum(features * weights, axis=-1))[0]
    heat = tf.math.divide_no_nan(heat, tf.reduce_max(heat))
    return tf.image.resize(heat[..., None], IMAGE_SIZE)[..., 0].numpy()


def evaluate(args):
    model, architecture = load_checkpoint(args.checkpoint)
    rows = read_splits(args.splits)
    selected = [r for r in rows if r["split"] == "test"]
    data = loader(args.data, rows, "test", args.batch_size, architecture)
    truth, predicted, confidences = [], [], []
    for images, labels in data:
        probs = tf.nn.softmax(model(images, training=False), axis=-1).numpy()
        truth.extend(labels.numpy().tolist())
        predicted.extend(probs.argmax(axis=1).tolist())
        confidences.extend(probs.max(axis=1).tolist())
    # tf.data giữ thứ tự manifest khi không shuffle tập test.
    errors = [(confidence, row["path"], actual, pred)
              for row, actual, pred, confidence in zip(selected, truth, predicted, confidences)
              if actual != pred]
    out = Path(args.output)
    out.mkdir(parents=True, exist_ok=True)
    report = classification_report(truth, predicted, labels=list(range(len(CLASSES))), target_names=CLASSES, output_dict=True, zero_division=0)
    (out / "metrics.json").write_text(json.dumps({"accuracy": accuracy_score(truth, predicted), "report": report}, indent=2), encoding="utf-8")
    fig, ax = plt.subplots(figsize=(10, 9))
    ConfusionMatrixDisplay.from_predictions(truth, predicted, labels=list(range(len(CLASSES))), display_labels=CLASSES, xticks_rotation=90, ax=ax, colorbar=False)
    fig.tight_layout()
    fig.savefig(out / "confusion_matrix.png", dpi=160)
    plt.close(fig)
    errors.sort(reverse=True)
    with (out / "errors.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["confidence", "path", "true", "predicted"])
        writer.writerows((c, p, CLASSES[t], CLASSES[v]) for c, p, t, v in errors)
    if errors:
        selected_errors = errors[:10]
        fig, axes = plt.subplots(2, 5, figsize=(18, 8), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, (confidence, path, actual, pred) in zip(axes.flat, selected_errors):
            with Image.open(Path(args.data) / path) as im:
                image = image_array(im).astype(np.uint8)
            ax.imshow(image)
            ax.set_title(f"{CLASSES[actual]} → {CLASSES[pred]} ({confidence:.2f})")
        fig.tight_layout()
        fig.savefig(out / "top_errors.png", dpi=160)
        plt.close(fig)
        with Image.open(Path(args.data) / selected_errors[0][1]) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
        heat = gradcam(model, architecture, image)
        fig, ax = plt.subplots()
        ax.imshow(image_array(image).astype(np.uint8))
        ax.imshow(heat, cmap="jet", alpha=.45, vmin=0, vmax=1)
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(out / "gradcam.png", dpi=160)
        plt.close(fig)
    else:
        for filename in ("top_errors.png", "gradcam.png"):
            (out / filename).unlink(missing_ok=True)
    print(f"Test accuracy: {accuracy_score(truth, predicted):.4f}; lỗi: {len(errors)}. Kết quả: {out}")

In [ ]:
for name in ("custom", "resnet50"):
    print("Đánh giá:", name)
    evaluate(SimpleNamespace(data=DATA_DIR, splits=SPLITS, batch_size=BATCH_SIZE,
                             checkpoint=RUNS / name / "best.keras",
                             output=RUNS / name / "test"))
    metrics = json.loads((RUNS / name / "test" / "metrics.json").read_text(encoding="utf-8"))
    print("Accuracy:", round(metrics["accuracy"], 4),
          "| Macro F1:", round(metrics["report"]["macro avg"]["f1-score"], 4),
          "| Weighted F1:", round(metrics["report"]["weighted avg"]["f1-score"], 4))
    display(DisplayImage(filename=str(RUNS / name / "test" / "confusion_matrix.png")))

## 8. Phân tích lỗi và Grad-CAM

Mã lưu toàn bộ mẫu dự đoán sai vào `errors.csv` và vẽ tối đa 10 ảnh sai có xác suất dự đoán cao nhất. Xem từng ảnh để cân nhắc che khuất, ảnh mờ, nhiều đối tượng hoặc nhãn gốc sai; không thể xác định nguyên nhân chỉ từ xác suất. Xác suất softmax cao cũng không bảo đảm mô hình được hiệu chuẩn tốt.

Grad-CAM lấy gradient của logit lớp được dự đoán theo đặc trưng tích chập cuối, lấy trung bình gradient làm trọng số, sau đó phủ heatmap lên ảnh. Đây là công cụ gợi ý vùng ảnh có ảnh hưởng cục bộ đến dự đoán, không chứng minh quan hệ nhân quả hoặc mô hình thực sự hiểu đối tượng.

In [ ]:
for name in ("custom", "resnet50"):
    folder = RUNS / name / "test"
    print("\nPhân tích", name)
    with (folder / "errors.csv").open(encoding="utf-8", newline="") as f:
        has_errors = next(csv.DictReader(f), None) is not None
    for filename in ("top_errors.png", "gradcam.png"):
        path = folder / filename
        if has_errors and path.exists():
            print(filename)
            display(DisplayImage(filename=str(path)))
        else:
            print(f"Không có {filename} (có thể không có mẫu dự đoán sai).")

## 9. Demo tải ảnh dự đoán top 3

Chọn `best.keras` và giữ `metadata.json` cùng thư mục sau khi huấn luyện. Gradio gọi Keras với `training=False`, dùng đúng preprocessing theo kiến trúc và softmax để hiển thị ba nhãn cao nhất. Với ảnh ngoài 10 lớp, mô hình vẫn buộc chọn một nhãn. Chạy ô này khi cần mở demo; gọi `demo.close()` để dừng server.

In [ ]:
import gradio as gr

DEMO_MODEL = "resnet50"  # Hoặc "custom".
demo_model, demo_architecture = load_checkpoint(RUNS / DEMO_MODEL / "best.keras")


def predict(image):
    if image is None:
        raise gr.Error("Vui lòng chọn một ảnh.")
    probabilities = predict_probabilities(demo_model, demo_architecture, image)
    return {LABELS[CLASSES[i]]: float(probabilities[i]) for i in range(len(CLASSES))}


demo = gr.Interface(fn=predict, inputs=gr.Image(type="pil", label="Ảnh động vật"),
                    outputs=gr.Label(num_top_classes=3, label="Ba dự đoán"),
                    title="Phân loại ảnh động vật — TensorFlow CNN")
demo.launch()

## 10. Gợi ý viết báo cáo

1. Nêu bài toán, nguồn dữ liệu, số ảnh hợp lệ/ảnh lỗi và cách chia dữ liệu. Trình bày phân bố lớp, kích thước, rủi ro ảnh trùng và chất lượng nhãn.
2. Mô tả preprocessing, augmentation, kiến trúc CNN, số tham số, ResNet50, hai giai đoạn, optimizer, learning rate và phần cứng.
3. Dùng learning curves để phân tích hội tụ và overfitting; báo cáo accuracy, macro/weighted F1 và chỉ số từng lớp trên cùng tập test.
4. Chọn vài mẫu sai có xác suất cao, giải thích dựa trên nội dung ảnh và đối chiếu Grad-CAM. Phân biệt quan sát với giả thuyết.
5. Nêu giới hạn: dữ liệu có thể trùng, nhãn có thể sai, tập test cùng nguồn với train, ảnh resize bị méo, mô hình không có lớp “không thuộc 10 loài”. Đề xuất đánh giá thêm trên dữ liệu độc lập.